# 06 — End-to-End Showcase: P.U.L.S.E.

**P.U.L.S.E. — Predictive Unified Life-sciences Summarization Engine**

This capstone notebook demonstrates the full platform in a single narrative:

| Step | Component | Output |
|------|-----------|--------|
| 1 | Feature Store | 127 k-row master table |
| 2 | Diabetes model | Risk score + SHAP |
| 3 | Cardio model | Probability + CI |
| 4 | Drift detector | CRITICAL / WARNING alerts |
| 5 | Drug RAG | Mistral clinical summary |
| 6 | MLflow | Experiment comparison |

---
**Clinical Scenario**: A 58-year-old male presents with BMI 31, fasting glucose 138 mg/dL, HbA1c 7.2%, BP 145/92 mmHg.
We assess diabetes risk, cardiovascular risk, check for recent model drift, and retrieve drug evidence for Metformin.

In [ ]:
import os, sys
ROOT = os.path.dirname(os.path.abspath('.'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings
warnings.filterwarnings('ignore')

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import shap

sns.set_theme(style='whitegrid', palette='muted')
MODEL_DIR = os.path.join(ROOT, 'models')
print('P.U.L.S.E. ready')

## Patient Profile

In [ ]:
patient = {
    'age':               58,
    'gender':             1,   # 1=Male (NHANES)
    'bmi':             31.0,
    'glucose':         138.0,
    'hba1c':             7.2,
    'blood_pressure_sys': 145.0,
    'blood_pressure_dia':  92.0,
}

flags = []
if patient['glucose'] >= 126:  flags.append('⚠️  Fasting glucose ≥ 126 → Diabetic range')
if patient['hba1c']  >= 6.5:   flags.append('⚠️  HbA1c ≥ 6.5% → Diabetic')
if patient['bmi']    >= 30:    flags.append('⚠️  BMI ≥ 30 → Obese')
if patient['blood_pressure_sys'] >= 140: flags.append('⚠️  Systolic BP ≥ 140 → Stage 2 hypertension')

print('--- Clinical Flags ---')
for f in flags: print(f)
print(f'\nTotal red flags: {len(flags)}')

## Step 1 — Diabetes Risk Assessment

In [ ]:
with open(os.path.join(MODEL_DIR, 'nhanes_diab_xgb.pkl'), 'rb') as f:
    diab_model = pickle.load(f)
with open(os.path.join(MODEL_DIR, 'nhanes_diab_features.pkl'), 'rb') as f:
    diab_features = pickle.load(f)

X_pat = pd.DataFrame([{k: patient[k] for k in diab_features}])
diab_prob = float(diab_model.predict_proba(X_pat)[0, 1])

tier = 'HIGH RISK 🔴' if diab_prob >= 0.70 else 'MODERATE 🟠' if diab_prob >= 0.40 else 'LOW RISK 🟢'
print(f'Diabetes probability : {diab_prob:.1%}')
print(f'Risk tier            : {tier}')

In [ ]:
# SHAP waterfall for this patient
explainer = shap.TreeExplainer(diab_model.get_booster())
shap_val  = explainer.shap_values(X_pat)

fig, ax = plt.subplots(figsize=(8, 4))
sort_idx = np.argsort(np.abs(shap_val[0]))[::-1]
sorted_feats  = [diab_features[i] for i in sort_idx]
sorted_shap   = shap_val[0][sort_idx]
colours       = ['#D32F2F' if v > 0 else '#1976D2' for v in sorted_shap]
ax.barh(sorted_feats[::-1], sorted_shap[::-1], color=colours[::-1])
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('SHAP value')
ax.set_title(f'SHAP Explanation — Patient  (P(diabetes)={diab_prob:.2%})')
plt.tight_layout()
plt.show()

## Step 2 — Cardiovascular Risk Assessment

In [ ]:
with open(os.path.join(MODEL_DIR, 'cardio_xgb.pkl'), 'rb') as f:
    cardio_model = pickle.load(f)
with open(os.path.join(MODEL_DIR, 'cardio_features.pkl'), 'rb') as f:
    cardio_feats = pickle.load(f)

# Representative high-risk Statlog profile matching patient characteristics
cardio_patient = {
    'age': 58, 'sex': 1, 'cp': 3, 'trestbps': 145,
    'chol': 280, 'fbs': 1, 'restecg': 1, 'thalach': 130,
    'exang': 1, 'oldpeak': 2.3, 'slope': 2, 'ca': 1, 'thal': 7
}

Xc = pd.DataFrame([[cardio_patient[f] for f in cardio_feats]], columns=cardio_feats)
cp = float(cardio_model.predict_proba(Xc)[0, 1])

print(f'Cardiovascular probability: {cp:.1%}')
print(f'Risk tier: {"HIGH RISK 🔴" if cp >= 0.5 else "LOW RISK 🟢"}')

## Step 3 — Drift & Model Health Check

In [ ]:
from src.monitoring.drift_detector import run_drift_report
from src.monitoring.health_report  import build_health_report, console_render

result = run_drift_report()
report = build_health_report(result)
console_render(report)

## Step 4 — Drug Evidence: Metformin for Type 2 Diabetes

In [ ]:
from src.nlp.summarizer import summarize_drug_no_llm

# Use retrieval-only (fast) — change to summarize_drug() for full LLM output
evidence = summarize_drug_no_llm(
    condition='Type 2 Diabetes',
    drug='Metformin',
    top_k=8,
)

print(evidence['summary'])

## Step 5 — Composite Risk Dashboard (matplotlib)

In [ ]:
fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

def gauge(ax, value, label, thresholds=(0.40, 0.70)):
    colour = ('#D32F2F' if value >= thresholds[1]
              else '#F57C00' if value >= thresholds[0]
              else '#388E3C')
    bar = ax.barh([''], [value], color=colour, height=0.5)
    ax.barh([''], [1.0], color='#EEEEEE', height=0.5, zorder=0)
    ax.set_xlim(0, 1)
    ax.set_xticks([0, 0.25, 0.50, 0.75, 1.0])
    ax.set_xticklabels(['0%', '25%', '50%', '75%', '100%'])
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.text(min(value + 0.03, 0.92), 0, f'{value:.1%}',
            va='center', fontsize=13, color=colour, fontweight='bold')
    ax.get_yaxis().set_visible(False)

gauge(fig.add_subplot(gs[0]), diab_prob, 'Diabetes Risk')
gauge(fig.add_subplot(gs[1]), cp,        'Cardiovascular Risk')

ax_d = fig.add_subplot(gs[2])
drift_items = report.get('feature_drift', {})
if drift_items:
    feats = list(drift_items.keys())
    dists = [drift_items[f]['distance'] for f in feats]
    cols  = ['#D32F2F' if drift_items[f]['drifted'] else '#388E3C' for f in feats]
    ax_d.barh(feats, dists, color=cols)
    ax_d.axvline(0.10, color='orange', ls='--', lw=1)
    ax_d.axvline(0.20, color='red',    ls='--', lw=1)
    ax_d.set_title('Feature Drift', fontsize=11, fontweight='bold')
    ax_d.set_xlabel('Distance')
else:
    ax_d.text(0.5, 0.5, 'No drift data', transform=ax_d.transAxes, ha='center')

fig.suptitle(
    'P.U.L.S.E.  —  Composite Risk Dashboard',
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()

## Step 6 — MLflow Experiment Log

In [ ]:
import mlflow

mlflow.set_tracking_uri(f'sqlite:///{ROOT}/mlflow.db')

runs = mlflow.search_runs(order_by=['metrics.auc DESC'])
if runs.empty:
    print('No MLflow runs found — run the model training scripts first.')
else:
    cols_show = ['tags.mlflow.runName', 'metrics.auc', 'metrics.f1',
                 'params.n_estimators', 'params.max_depth']
    cols_show = [c for c in cols_show if c in runs.columns]
    print(runs[cols_show].head(10).to_string(index=False))